# Seleção e estatísticas de ordem — Tutorial

**Algoritmos e Estruturas de Dados II (COMP0498) — UFS — 2026.2**

Este tutorial retoma os algoritmos vistos em aula e pede que você os implemente, meça e
quebre. Todo o código é em **C**; as células usam `%%writefile` para gravar o arquivo e
`!gcc` para compilar e executar.

## Objetivos

Ao final deste tutorial você será capaz de:

- Implementar a busca do **menor** e do **segundo menor** contando as comparações de fato realizadas;
- Escrever duas soluções de **força bruta** para o $k$-ésimo menor e explicar por que cada uma é quadrática;
- Implementar a **partição de Lomuto** e o **quickselect**, nas versões recursiva e iterativa;
- **Medir** empiricamente a diferença entre força bruta e quickselect, e reconhecer a assinatura de $\Theta(n)$ e $\Theta(n^2)$ num gráfico;
- Construir a entrada que força o **pior caso** e mostrar que o **pivô aleatório** a neutraliza;
- Implementar a **mediana das medianas** e verificar a garantia de $7n/10$.

> **Como usar:** execute as células na ordem. Onde houver `TODO`, a célula seguinte compila e
> testa sua implementação — ela deve imprimir `OK`.

In [ ]:
!gcc --version | head -1

## O vetor de trabalho

Ao longo do tutorial usamos o mesmo vetor da aula:

```
A = {31, 8, 17, 4, 25, 12, 20, 6, 29}     (n = 9)
```

Ordenado, ele é `{4, 6, 8, 12, 17, 20, 25, 29, 31}`. Logo:
a $1^{a}$ estatística de ordem é $4$, a $2^{a}$ é $6$, a mediana ($k=5$) é $17$ e a $9^{a}$ é $31$.
Guarde esses valores — eles são o gabarito de quase todos os testes daqui para a frente.

## 1. O menor e o segundo menor

O mínimo custa exatamente $n-1$ comparações. Já o segundo menor tem duas soluções: fazer
**duas passadas** ($2n-3$ comparações) ou manter **dois candidatos** numa passada só.

O programa abaixo implementa as duas e **conta as comparações realmente executadas**, para
você confrontar a teoria com o número impresso.

In [ ]:
%%writefile menor.c
#include <stdio.h>

static long comps;   /* contador global de comparacoes entre elementos */

/* Indice do menor de A[e..d]. */
int indice_do_menor(const int A[], int e, int d) {
    int menor = e;
    for (int i = e + 1; i <= d; i++) {
        comps++;
        if (A[i] < A[menor]) menor = i;
    }
    return menor;
}

/* Segundo menor em DUAS passadas: acha o menor, ignora-o, acha o menor de novo. */
int segundo_menor_duas_passadas(const int A[], int n) {
    int im = indice_do_menor(A, 0, n - 1);
    int segundo = -1;
    for (int i = 0; i < n; i++) {
        if (i == im) continue;
        if (segundo == -1) { segundo = A[i]; continue; }   /* inicializa: sem comparar */
        comps++;
        if (A[i] < segundo) segundo = A[i];
    }
    return segundo;
}

/* Segundo menor em UMA passada: mantem os dois melhores candidatos. */
int segundo_menor_uma_passada(const int A[], int n) {
    int m1 = A[0] < A[1] ? A[0] : A[1];   /* menor  */
    int m2 = A[0] < A[1] ? A[1] : A[0];   /* segundo */
    comps++;
    for (int i = 2; i < n; i++) {
        comps++;
        if (A[i] < m1) { m2 = m1; m1 = A[i]; }
        else { comps++; if (A[i] < m2) m2 = A[i]; }
    }
    return m2;
}

int main(void) {
    int A[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int n = sizeof(A) / sizeof(A[0]);

    comps = 0;
    int menor = A[indice_do_menor(A, 0, n - 1)];
    printf("menor            = %2d   (%ld comparacoes, esperado n-1 = %d)\n",
           menor, comps, n - 1);

    comps = 0;
    int s2 = segundo_menor_duas_passadas(A, n);
    printf("2o menor (2 pas.)= %2d   (%ld comparacoes, esperado 2n-3 = %d)\n",
           s2, comps, 2 * n - 3);

    comps = 0;
    int s1 = segundo_menor_uma_passada(A, n);
    printf("2o menor (1 pas.)= %2d   (%ld comparacoes)\n", s1, comps);
    return 0;
}

In [ ]:
!gcc -Wall -O2 menor.c -o menor && ./menor

### Pergunta 1

Compare os três números impressos. As duas passadas gastam exatamente $2n-3 = 15$ comparações,
como previsto. A versão de uma passada gasta um pouco menos — mas **não** chega ao limite do
torneio, que para $n = 9$ seria $n + \lceil \log_2 n \rceil - 2 = 11$.

Por quê? Porque manter dois candidatos **não é** o torneio: é apenas uma reorganização das duas
passadas. No pior caso ela também faz $2n-3$ comparações (toda vez que $A[i] \ge m_1$, ela gasta
uma segunda comparação contra $m_2$). Para chegar ao limite do torneio é preciso guardar
*quem perdeu para quem*, o que exige memória extra $O(n)$.

**Exercício 1.** Implemente o segundo menor **pelo torneio**, com um vetor auxiliar que registre,
para cada elemento, a lista de adversários que ele venceu. Confira que o número de comparações
fica em $n + \lceil \log_2 n \rceil - 2$.

In [ ]:
%%writefile ex1_torneio.c
#include <stdio.h>
#include <stdlib.h>

static long comps;

/* TODO: implemente o segundo menor pelo torneio.
   Sugestao de estrutura:
     - mantenha um vetor 'atual' com os competidores da rodada;
     - para cada elemento i, mantenha 'perdedores[i]' = quem perdeu para ele;
     - ao final, o segundo menor e' o menor dentre os perdedores do campeao.
   Nao esqueca de incrementar 'comps' a cada comparacao entre elementos. */
int segundo_menor_torneio(const int A[], int n) {
    /* TODO: implemente aqui */
    (void)A; (void)n;
    return -1;
}

int main(void) {
    int A[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int n = sizeof(A) / sizeof(A[0]);

    comps = 0;
    int r = segundo_menor_torneio(A, n);
    int limite = n + 4 - 2;          /* n + teto(log2 9) - 2 = 9 + 4 - 2 */

    printf("resultado = %d (esperado 6), comparacoes = %ld (limite %d)\n",
           r, comps, limite);
    if (r == 6 && comps <= limite) printf("OK\n");
    else printf("FALHOU\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 ex1_torneio.c -o ex1_torneio && ./ex1_torneio

## 2. Força bruta para o $k$-ésimo menor

Duas estratégias, ambas quadráticas para a mediana:

1. **Contagem** — para cada $x$, conte quantos elementos são menores que ele; se forem $k-1$,
   então $x$ é a resposta. Custa $\Theta(n^2)$ para *qualquer* $k$.
2. **Seleção repetida** — ache o menor $k$ vezes, riscando o escolhido. Custa $\Theta(kn)$:
   barato para $k$ pequeno, quadrático para $k \approx n/2$.

O programa abaixo implementa as duas e mostra o custo de cada uma em função de $k$.

In [ ]:
%%writefile forca_bruta.c
#include <stdio.h>
#include <string.h>

static long comps;

/* Estrategia 1: contagem. Assume elementos DISTINTOS. */
int por_contagem(const int A[], int n, int k) {
    for (int i = 0; i < n; i++) {
        int menores = 0;
        for (int j = 0; j < n; j++) {
            comps++;
            if (A[j] < A[i]) menores++;
        }
        if (menores == k - 1) return A[i];
    }
    return -1;
}

/* Estrategia 2: selecao repetida (selection sort interrompido). */
int por_selecao_repetida(const int A[], int n, int k) {
    int usado[256] = {0};
    int resposta = -1;
    for (int r = 0; r < k; r++) {
        int melhor = -1;
        for (int i = 0; i < n; i++) {
            if (usado[i]) continue;
            if (melhor == -1) { melhor = i; continue; }
            comps++;
            if (A[i] < A[melhor]) melhor = i;
        }
        usado[melhor] = 1;
        resposta = A[melhor];
    }
    return resposta;
}

int main(void) {
    int A[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int n = sizeof(A) / sizeof(A[0]);

    printf(" k | contagem            | selecao repetida\n");
    printf("---+---------------------+------------------\n");
    for (int k = 1; k <= n; k++) {
        comps = 0; int a = por_contagem(A, n, k);        long ca = comps;
        comps = 0; int b = por_selecao_repetida(A, n, k); long cb = comps;
        printf("%2d | %2d  (%3ld comparacoes) | %2d  (%3ld comparacoes)\n", k, a, ca, b, cb);
    }
    return 0;
}

In [ ]:
!gcc -Wall -O2 forca_bruta.c -o forca_bruta && ./forca_bruta

### Pergunta 2

Olhe a coluna da **contagem**: o custo pula de um lado para o outro sem seguir $k$ — $36$, $72$,
$18$, $54$\dots\ Isso porque a função para assim que encontra a resposta, então ela gasta
$n \times (\text{posição da resposta no vetor original})$ comparações. Como essa posição não tem
nenhuma relação com $k$, o custo parece aleatório. No **pior caso** (resposta na última posição)
são $n^2 = 81$ comparações, e na média $n^2/2$: quadrático de qualquer jeito.

Já a **seleção repetida** cresce de forma limpa com $k$ — $8$, $15$, $21$, $26$, $30$\dots\ —
somando $(n-1) + (n-2) + \cdots$. É a assinatura do $\Theta(kn)$, e é por isso que ela é ótima
para $k$ pequeno e péssima para a mediana.

**Exercício 2.** O `por_contagem` devolve `-1` se houver elementos repetidos —
com `A = {5, 5, 1}` e `k = 2`, nenhum elemento tem exatamente 1 menor que ele.
Conserte a função para tratar repetições corretamente.

*Dica:* conte também quantos elementos **iguais** aparecem **antes** de `A[i]`, e teste
`menores + iguais_antes == k - 1`. Isso desempata as cópias atribuindo a cada uma um posto
distinto.

In [ ]:
%%writefile ex2_repetidos.c
#include <stdio.h>

/* TODO: conserte esta funcao para funcionar com elementos repetidos. */
int por_contagem(const int A[], int n, int k) {
    for (int i = 0; i < n; i++) {
        int menores = 0;
        for (int j = 0; j < n; j++)
            if (A[j] < A[i]) menores++;
        if (menores == k - 1) return A[i];   /* TODO: ajuste esta condicao */
    }
    return -1;
}

int main(void) {
    int A[] = {5, 5, 1};
    int B[] = {7, 7, 7, 7};
    int C[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int ok = 1;

    ok &= (por_contagem(A, 3, 1) == 1);
    ok &= (por_contagem(A, 3, 2) == 5);
    ok &= (por_contagem(A, 3, 3) == 5);
    ok &= (por_contagem(B, 4, 3) == 7);
    ok &= (por_contagem(C, 9, 5) == 17);   /* nao pode quebrar o caso distinto */

    printf(ok ? "OK\n" : "FALHOU\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 ex2_repetidos.c -o ex2_repetidos && ./ex2_repetidos

## 3. A partição de Lomuto

A partição é a única ferramenta nova do quickselect. Ela escolhe um pivô, rearranja o trecho e
devolve o índice $q$ onde o pivô parou — **sua posição final**.

O invariante do laço é:

$$A[e..i] \le \text{pivô} \qquad\text{e}\qquad A[i+1..j-1] > \text{pivô}$$

O programa abaixo imprime o vetor a cada iteração para você ver o invariante se manter.

In [ ]:
%%writefile particiona.c
#include <stdio.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

static void imprime(const int A[], int n, int i, int j, int d) {
    for (int t = 0; t < n; t++) {
        char esq = (t == i + 1) ? '[' : ' ';
        printf("%c%2d%c", esq, A[t], (t == j) ? '<' : (t == d ? '*' : ' '));
    }
    printf("   (i=%d, j=%d)\n", i, j);
}

int particiona(int A[], int e, int d) {
    int pivo = A[d];
    int i = e - 1;
    printf("pivo = %d   ('*' marca o pivo, '<' marca A[j])\n", pivo);
    for (int j = e; j < d; j++) {
        if (A[j] <= pivo) { i++; troca(&A[i], &A[j]); }
        imprime(A, d + 1, i, j, d);
    }
    troca(&A[i + 1], &A[d]);
    return i + 1;
}

int main(void) {
    int A[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int n = sizeof(A) / sizeof(A[0]);

    int q = particiona(A, 0, n - 1);
    printf("\nq = %d, A[q] = %d\n", q, A[q]);
    printf("vetor final: ");
    for (int t = 0; t < n; t++) printf("%d ", A[t]);
    printf("\n");
    printf("posto do pivo = q - e + 1 = %d\n", q - 0 + 1);
    return 0;
}

In [ ]:
!gcc -Wall -O2 particiona.c -o particiona && ./particiona

### Pergunta 3

Confira a saída: o pivô $29$ parou em $q = 7$, então ele é a $8^{a}$ estatística de ordem de $A$.
E de fato, no vetor ordenado `{4, 6, 8, 12, 17, 20, 25, 29, 31}`, o $29$ está na oitava posição.

**Essa é a observação que resolve o problema todo:** uma única partição, que custa $O(n)$, já
determina exatamente uma estatística de ordem — e ainda diz de que lado estão todas as outras.

## 4. Quickselect

Agora é só juntar as peças. Depois de particionar, compare $k$ com o posto do pivô:

| Se… | então… |
|---|---|
| $k = \text{posto}$ | o pivô é a resposta |
| $k < \text{posto}$ | recorra à **esquerda**, com o mesmo $k$ |
| $k > \text{posto}$ | recorra à **direita**, com $k \leftarrow k - \text{posto}$ |

O programa abaixo testa **todos** os $k$ de $1$ a $n$ contra o vetor ordenado.

In [ ]:
%%writefile quickselect.c
#include <stdio.h>
#include <string.h>

static long comps;

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

int particiona(int A[], int e, int d) {
    int pivo = A[d], i = e - 1;
    for (int j = e; j < d; j++) {
        comps++;
        if (A[j] <= pivo) { i++; troca(&A[i], &A[j]); }
    }
    troca(&A[i + 1], &A[d]);
    return i + 1;
}

/* k-esima estatistica de ordem (1-indexada) de A[e..d]. Reordena A. */
int quickselect(int A[], int e, int d, int k) {
    if (e == d) return A[e];
    int q = particiona(A, e, d);
    int posto = q - e + 1;
    if (k == posto)      return A[q];
    else if (k < posto)  return quickselect(A, e, q - 1, k);
    else                 return quickselect(A, q + 1, d, k - posto);
}

int main(void) {
    int base[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int gab[]  = { 4, 6,  8, 12, 17, 20, 25, 29, 31};   /* base ordenado */
    int n = sizeof(base) / sizeof(base[0]);
    int ok = 1;

    for (int k = 1; k <= n; k++) {
        int A[9];
        memcpy(A, base, sizeof(base));   /* quickselect destroi a ordem: copie */
        comps = 0;
        int r = quickselect(A, 0, n - 1, k);
        printf("k=%d -> %2d (esperado %2d)  %3ld comparacoes\n", k, r, gab[k-1], comps);
        if (r != gab[k-1]) ok = 0;
    }
    printf(ok ? "OK\n" : "FALHOU\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 quickselect.c -o quickselect && ./quickselect

### Exercício 3 — versão iterativa

A recursão do quickselect é sempre **de cauda**: há uma única chamada e ela é a última coisa
que a função faz. Isso significa que ela vira um laço `while` mecanicamente, e o algoritmo passa
a usar espaço auxiliar $O(1)$.

Implemente `quickselect_iter` sem recursão. **Cuidado com o desconto de $k$** ao ir para a direita
— esquecê-lo é o erro clássico, e o programa não trava: só devolve o elemento errado.

In [ ]:
%%writefile ex3_iterativo.c
#include <stdio.h>
#include <string.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

int particiona(int A[], int e, int d) {
    int pivo = A[d], i = e - 1;
    for (int j = e; j < d; j++)
        if (A[j] <= pivo) { i++; troca(&A[i], &A[j]); }
    troca(&A[i + 1], &A[d]);
    return i + 1;
}

/* TODO: implemente a versao iterativa. Use um laco 'while (e < d)' e
   atualize e, d e k conforme a comparacao entre k e o posto do pivo. */
int quickselect_iter(int A[], int n, int k) {
    int e = 0, d = n - 1;
    /* TODO: implemente aqui */
    (void)e; (void)d; (void)k;
    return -1;
}

int main(void) {
    int base[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int gab[]  = { 4, 6,  8, 12, 17, 20, 25, 29, 31};
    int n = 9, ok = 1;

    for (int k = 1; k <= n; k++) {
        int A[9];
        memcpy(A, base, sizeof(base));
        int r = quickselect_iter(A, n, k);
        if (r != gab[k-1]) { printf("k=%d devolveu %d, esperado %d\n", k, r, gab[k-1]); ok = 0; }
    }
    printf(ok ? "OK\n" : "FALHOU\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 ex3_iterativo.c -o ex3_iterativo && ./ex3_iterativo

## 5. Medindo: força bruta contra quickselect

Teoria é uma coisa; a curva medida é outra. O programa `bench.c` recebe quatro argumentos:

```
./bench  n  k  alg  tipo  [semente]
```

- `alg`: `0` = força bruta por contagem, `1` = quickselect (pivô `A[d]`), `2` = quickselect com pivô aleatório
- `tipo`: `0` = vetor embaralhado, `1` = vetor **já ordenado**
- um quinto argumento opcional fixa a **semente** (padrão `12345`), para podermos repetir a
  medição com instâncias diferentes e tirar médias

Ele imprime o resultado e o número de comparações. Vamos usá-lo para desenhar as curvas.

In [ ]:
%%writefile bench.c
#include <stdio.h>
#include <stdlib.h>

static long comps;

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

/* Sem saida antecipada: medimos o custo de PIOR CASO, que e' o que a analise preve.
   (Com 'return' assim que acha, o custo passaria a depender de onde a resposta
   calhou de estar no vetor, e a curva viraria ruido.) */
int por_contagem(const int A[], int n, int k) {
    int resposta = -1;
    for (int i = 0; i < n; i++) {
        int menores = 0;
        for (int j = 0; j < n; j++) { comps++; if (A[j] < A[i]) menores++; }
        if (menores == k - 1) resposta = A[i];
    }
    return resposta;
}

int particiona(int A[], int e, int d) {
    int pivo = A[d], i = e - 1;
    for (int j = e; j < d; j++) { comps++; if (A[j] <= pivo) { i++; troca(&A[i], &A[j]); } }
    troca(&A[i + 1], &A[d]);
    return i + 1;
}

int quickselect_iter(int A[], int n, int k, int aleatorio) {
    int e = 0, d = n - 1;
    while (e < d) {
        if (aleatorio) troca(&A[e + rand() % (d - e + 1)], &A[d]);
        int q = particiona(A, e, d);
        int posto = q - e + 1;
        if (k == posto) return A[q];
        if (k < posto) d = q - 1;
        else { k -= posto; e = q + 1; }
    }
    return A[e];
}

int main(int argc, char **argv) {
    if (argc != 5 && argc != 6) {
        fprintf(stderr, "uso: %s n k alg tipo [semente]\n", argv[0]); return 1;
    }
    int n = atoi(argv[1]), k = atoi(argv[2]);
    int alg = atoi(argv[3]), tipo = atoi(argv[4]);
    unsigned semente = (argc == 6) ? (unsigned)atoi(argv[5]) : 12345u;

    int *A = malloc((size_t)n * sizeof(int));
    for (int i = 0; i < n; i++) A[i] = i + 1;      /* valores distintos 1..n */

    srand(semente);                                /* reprodutivel, mas variavel */
    if (tipo == 0)                                 /* embaralha (Fisher-Yates) */
        for (int i = n - 1; i > 0; i--) troca(&A[i], &A[rand() % (i + 1)]);

    comps = 0;
    int r = (alg == 0) ? por_contagem(A, n, k)
                       : quickselect_iter(A, n, k, alg == 2);

    /* como os valores sao 1..n, a k-esima estatistica de ordem e' exatamente k */
    printf("%d %ld %s\n", r, comps, (r == k) ? "ok" : "ERRO");
    free(A);
    return 0;
}

In [ ]:
!gcc -Wall -O2 bench.c -o bench && ./bench 9 5 1 0 && ./bench 9 5 0 0

### O gráfico

Vamos medir a mediana ($k = \lceil n/2 \rceil$) para vários $n$, com os três algoritmos, e plotar
o número de comparações. Repare no eixo: a força bruta explode enquanto as duas versões do
quickselect ficam coladas numa reta.

In [ ]:
import subprocess
from statistics import mean
import matplotlib.pyplot as plt

def comparacoes(n, k, alg, tipo=0, semente=12345):
    saida = subprocess.run(["./bench", str(n), str(k), str(alg), str(tipo), str(semente)],
                           capture_output=True, text=True).stdout.split()
    assert saida[2] == "ok", f"bench devolveu resultado errado: {saida}"
    return int(saida[1])

# uma única execução varia MUITO (o pivô depende do sorteio); tiramos a média
SEMENTES = range(1, 26)

def media(n, alg, tipo=0):
    k = (n + 1) // 2                       # a mediana
    return mean(comparacoes(n, k, alg, tipo, s) for s in SEMENTES)

ns_bruta = [200, 400, 800, 1600, 3200]
ns_quick = [200, 400, 800, 1600, 3200, 6400, 12800, 25600]

bruta = [media(n, 0) for n in ns_bruta]
quick = [media(n, 1) for n in ns_quick]
alea  = [media(n, 2) for n in ns_quick]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(ns_bruta, bruta, "o-", label="força bruta (contagem)")
ax[0].plot(ns_quick, quick, "s-", label="quickselect")
ax[0].plot(ns_quick, alea,  "^-", label="quickselect aleatório")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_title("comparações para achar a mediana (log–log)")
ax[0].set_xlabel("n"); ax[0].set_ylabel("comparações"); ax[0].legend()

ax[1].plot(ns_quick, [c / n for c, n in zip(quick, ns_quick)], "s-", label="quickselect")
ax[1].plot(ns_quick, [c / n for c, n in zip(alea,  ns_quick)], "^-", label="quickselect aleatório")
ax[1].axhline(3.39, ls="--",  c="gray",  label="esperança teórica ≈ 3,39n")
ax[1].axhline(2.00, ls=":",   c="black", label="caso ideal (partição perfeita) 2n")
ax[1].set_ylim(0, 6)
ax[1].set_title("comparações / n  —  se for Θ(n), esta curva é plana")
ax[1].set_xscale("log")
ax[1].set_xlabel("n"); ax[1].set_ylabel("comparações / n"); ax[1].legend()

plt.tight_layout(); plt.show()

### Pergunta 4

O gráfico da esquerda está em escala **log–log**: nela, uma potência $n^p$ vira uma reta de
inclinação $p$. A força bruta sobe com inclinação $2$; as duas versões do quickselect, com
inclinação $1$. É a leitura visual de $\Theta(n^2)$ contra $\Theta(n)$.

O gráfico da direita é ainda mais direto: dividir o número de comparações por $n$ **achata** a
curva se o algoritmo for linear. E é exatamente o que acontece — a razão oscila em torno de
$3{,}4$ e **não cresce** com $n$.

Mas repare: ela não fica em $2$. A conta $n + n/2 + n/4 + \cdots < 2n$ da aula supõe que **toda**
partição corta o trecho exatamente ao meio — é o caso ideal, não a média. Fazendo a esperança
sobre pivôs uniformes, o valor correto para a mediana é $\approx 3{,}39n$ comparações, que é a
linha tracejada. O caso ideal e a esperança diferem por uma constante; ambos são $\Theta(n)$,
e é só isso que a análise assintótica promete.

Se você repetisse o experimento com um algoritmo $\Theta(n \log n)$, a razão cresceria lentamente
(como $\log n$); com $\Theta(n^2)$, cresceria linearmente.

> **Por que a média sobre 25 sementes?** Uma execução isolada varia muito — o custo depende dos
> pivôs sorteados. Sem a média, a curva da direita fica serrilhada entre $1{,}8$ e $5{,}1$, e não
> dá para enxergar que ela é plana. Medir uma quantidade aleatória uma vez só não mede nada.

## 6. Quebrando o quickselect: o pior caso

O caso médio $\Theta(n)$ não é uma garantia. Com a partição de Lomuto usando `A[d]` como pivô,
um vetor **já ordenado** faz o pivô ser sempre o maior elemento: cada partição descarta um único
elemento, e o custo vira

$$T(n) = T(n-1) + cn = \Theta(n^2).$$

Vamos construir exatamente essa entrada (`tipo = 1`) e ver as duas curvas separadas.

In [ ]:
ns = [500, 1000, 2000, 4000, 8000]

fixo_ord = [comparacoes(n, 1, 1, tipo=1) for n in ns]   # pivô A[d], vetor ordenado
alea_ord = [comparacoes(n, 1, 2, tipo=1) for n in ns]   # pivô aleatório, mesmo vetor

print(f"{'n':>6} | {'pivô A[d]':>12} | {'n(n-1)/2':>12} | {'pivô aleatório':>14}")
print("-" * 54)
for n, a, b in zip(ns, fixo_ord, alea_ord):
    print(f"{n:>6} | {a:>12,} | {n*(n-1)//2:>12,} | {b:>14,}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ns, fixo_ord, "o-", label="pivô = A[d]  (vetor ordenado)")
ax.plot(ns, alea_ord, "^-", label="pivô aleatório (mesmo vetor)")
ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("comparações (escala log)")
ax.set_title("a mesma entrada, dois comportamentos")
ax.legend(); plt.tight_layout(); plt.show()

### Pergunta 5

A coluna do meio confirma a conta: com o vetor ordenado e $k = 1$, o pivô fixo faz exatamente
$n(n-1)/2$ comparações — o pior caso teórico, atingido em cheio.

O pivô aleatório resolve? **Em parte.** O pior caso $\Theta(n^2)$ continua existindo — o que muda
é que ele deixa de depender da *entrada* e passa a depender dos *sorteios*. Nenhum adversário
consegue construir um vetor ruim, porque não conhece os sorteios do seu programa. Mas se a sorte
for péssima, o quadrático acontece.

**Exercício 4.** Use o quinto argumento do `bench` para rodar `alg = 2` (pivô aleatório) sobre o
vetor ordenado com, digamos, 500 sementes diferentes, e registre o **pior** resultado observado.

1. Quão longe o pior caso observado fica de $n(n-1)/2$?
2. Faça um histograma dos 500 valores. A distribuição é simétrica? Onde fica a média em relação
   à mediana da distribuição?
3. Estime, a partir da sua amostra, a probabilidade de o custo passar de $10n$. Compare com a sua
   intuição antes de medir.

## 7. Mediana das medianas: $O(n)$ garantido

Para ter garantia no pior caso, escolhemos o pivô em vez de sorteá-lo:

1. divida em grupos de **5**;
2. ordene cada grupo e pegue sua mediana ($O(1)$ por grupo);
3. ache recursivamente a **mediana dessas medianas**, $x$;
4. particione em torno de $x$.

O ganho é a garantia de que ao menos $3n/10$ elementos ficam de cada lado de $x$, logo o trecho
sobrevivente tem no máximo $7n/10 + 6$ elementos. A recorrência

$$T(n) \le T(n/5) + T(7n/10) + cn$$

é linear porque $\tfrac{1}{5} + \tfrac{7}{10} = \tfrac{9}{10} < 1$.

In [ ]:
%%writefile ex4_mediana_medianas.c
#include <stdio.h>
#include <string.h>

static void troca(int *a, int *b) { int t = *a; *a = *b; *b = t; }

static void insertion_sort(int A[], int e, int d) {
    for (int i = e + 1; i <= d; i++) {
        int v = A[i], j = i - 1;
        while (j >= e && A[j] > v) { A[j + 1] = A[j]; j--; }
        A[j + 1] = v;
    }
}

int particiona(int A[], int e, int d) {
    int pivo = A[d], i = e - 1;
    for (int j = e; j < d; j++)
        if (A[j] <= pivo) { i++; troca(&A[i], &A[j]); }
    troca(&A[i + 1], &A[d]);
    return i + 1;
}

int selecao(int A[], int e, int d, int k);   /* declaracao antecipada */

/* TODO: devolva o INDICE de um pivo bom para A[e..d].
   Roteiro:
     1. se n = d-e+1 <= 5: insertion_sort(A, e, d) e devolva e + n/2;
     2. para cada grupo de 5 (i = e, e+5, e+10, ...):
          - ordene o grupo com insertion_sort;
          - troque a mediana do grupo para A[e + g], incrementando g;
     3. chame selecao(A, e, e+g-1, (g+1)/2) para achar a mediana das medianas;
     4. devolva o indice onde ela ficou: e + (g-1)/2. */
int pivo_mediana_das_medianas(int A[], int e, int d) {
    /* TODO: implemente aqui */
    (void)A; (void)e;
    return d;      /* provisorio: devolve A[d], ou seja, o quickselect comum */
}

int particiona_em(int A[], int e, int d, int p) {
    troca(&A[p], &A[d]);
    return particiona(A, e, d);
}

int selecao(int A[], int e, int d, int k) {
    if (e == d) return A[e];
    int p = pivo_mediana_das_medianas(A, e, d);
    int q = particiona_em(A, e, d, p);
    int posto = q - e + 1;
    if (k == posto)      return A[q];
    else if (k < posto)  return selecao(A, e, q - 1, k);
    else                 return selecao(A, q + 1, d, k - posto);
}

int main(void) {
    int base[] = {31, 8, 17, 4, 25, 12, 20, 6, 29};
    int gab[]  = { 4, 6,  8, 12, 17, 20, 25, 29, 31};
    int ok = 1;

    for (int k = 1; k <= 9; k++) {
        int A[9];
        memcpy(A, base, sizeof(base));
        int r = selecao(A, 0, 8, k);
        if (r != gab[k-1]) { printf("k=%d devolveu %d, esperado %d\n", k, r, gab[k-1]); ok = 0; }
    }

    /* teste maior: vetor 1..300 em ordem decrescente */
    int B[300], G[300];
    for (int i = 0; i < 300; i++) { B[i] = 300 - i; G[i] = i + 1; }
    for (int k = 1; k <= 300; k += 37) {
        int A[300];
        memcpy(A, B, sizeof(B));
        int r = selecao(A, 0, 299, k);
        if (r != G[k-1]) { printf("n=300 k=%d devolveu %d, esperado %d\n", k, r, G[k-1]); ok = 0; }
    }

    printf(ok ? "OK\n" : "FALHOU\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 ex4_mediana_medianas.c -o ex4_mm && ./ex4_mm

> **Atenção:** o esqueleto acima já passa nos testes *antes* de você implementar nada — porque
> `return d` reduz o algoritmo ao quickselect comum, que também está **correto**, só não tem a
> garantia de pior caso. Os testes verificam a **corretude**, não a complexidade.
>
> Para verificar a *garantia*, use o exercício abaixo.

### Exercício 5 — a garantia de $7n/10$

Instrumente sua implementação para registrar, a cada chamada de `selecao`, o tamanho do trecho
antes e depois da partição. Verifique que

$$\text{tamanho do trecho sobrevivente} \le \frac{7}{10} \cdot \text{tamanho atual} + 6$$

vale em **todas** as chamadas quando o pivô vem da mediana das medianas — e que ela é **violada**
com o pivô fixo `A[d]` sobre um vetor ordenado.

### Exercício 6 — grupos de 3 não funcionam

Nos slides afirmamos que grupos de $3$ **não** produzem um algoritmo linear. Justifique:

1. Refaça o argumento da figura para grupos de $3$: quantos elementos ficam garantidamente
   de cada lado do pivô? (Resposta: $2 \cdot \lceil n/6 \rceil \approx n/3$.)
2. Escreva a recorrência resultante.
3. Aplique o método da substituição com a hipótese $T(m) \le am$ e mostre exatamente onde ele
   falha — isto é, por que não existe $a$ que feche a indução.

*Dica:* some as frações dos dois subproblemas. O que acontece quando essa soma é igual a $1$?

## Desafio Final

Implemente o **introselect**: o híbrido que a `libstdc++` usa em `std::nth_element`.

**Especificação.**

```c
int introselect(int A[], int n, int k);
```

- Comece com o quickselect de **pivô aleatório**.
- Conte as rodadas de partição. Se o número de rodadas passar de $2\lceil \log_2 n \rceil$ sem
  encontrar a resposta, **troque** para a mediana das medianas no trecho que restou.
- O resultado deve ser correto para todo $1 \le k \le n$.

**O que entregar.**

1. A implementação, passando nos mesmos testes de corretude das seções anteriores.
2. Uma tabela comparando **comparações** e **tempo de parede** de quatro algoritmos —
   quickselect com pivô `A[d]`, quickselect aleatório, mediana das medianas pura e introselect —
   sobre duas famílias de entrada: vetor embaralhado e vetor já ordenado, com
   $n \in \{10^3, 10^4, 10^5, 10^6\}$ e $k = \lceil n/2 \rceil$.
3. Um parágrafo respondendo: **em qual das quatro colunas o introselect perde**, e por quê isso
   é um preço aceitável.

*Dica para o item 2:* meça tempo com `clock_gettime(CLOCK_MONOTONIC, ...)` e repita cada medição
algumas vezes, ficando com a mediana — apropriadamente, use o seu próprio quickselect para
calculá-la.

In [ ]:
%%writefile desafio_introselect.c
#include <stdio.h>
#include <stdlib.h>

/* TODO: resolva o desafio aqui.
   Reaproveite particiona(), pivo_mediana_das_medianas() e selecao()
   das secoes anteriores. */

int introselect(int A[], int n, int k) {
    /* TODO: implemente aqui */
    (void)A; (void)n; (void)k;
    return -1;
}

int main(void) {
    /* TODO: seus testes de corretude e sua tabela de medicoes */
    printf("implemente o desafio\n");
    return 0;
}

In [ ]:
!gcc -Wall -O2 desafio_introselect.c -o desafio && ./desafio

## Referências

- **CLRS**, *Algoritmos: Teoria e Prática*, 4ª ed. — Capítulo 9, "Medianas e estatísticas de ordem".
  A análise probabilística completa do quickselect está na seção 9.2 e a mediana das medianas na 9.3.
- **Levitin**, *Introdução à Análise e ao Projeto de Algoritmos*, 3ª ed. — Seção 5.6, que apresenta
  o quickselect sob a ótica da *redução do problema*.
- **Blum, Floyd, Pratt, Rivest, Tarjan (1973)**, "Time Bounds for Selection",
  *J. Computer and System Sciences* 7(4):448–461 — o artigo original da mediana das medianas.
- **Hoare (1961)**, "Algorithm 65: FIND", *CACM* 4(7):321–322 — o quickselect, publicado no mesmo
  ano do quicksort.

A lista completa em BibTeX está em `../referencias.bib`.